# train-eval-mode-branch — ex1: flip train and eval mode around dropout

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `train-eval-mode-branch`. Running the final beacon cell reports progress against the `PyTorch: train/eval mode` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: train/eval mode` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`train-eval-mode-branch`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "train-eval-mode-branch"
DD_SUBTOPIC = "PyTorch: train/eval mode"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `model.train()` vs `model.eval()` — quick refresher

Some layers behave DIFFERENTLY at train time vs eval time. Two big examples:

- **Dropout** drops activations only when `model.training is True`. In eval mode dropout is a no-op (identity).
- **BatchNorm** uses the current batch's mean/var in train mode and updates running statistics; in eval mode it uses the frozen running stats and updates nothing.

`model.train()` and `model.eval()` flip the `self.training` flag RECURSIVELY on every submodule. Forgetting `model.eval()` at validation time is the #1 reason ARENA learners see 'my validation accuracy is wildly noisy and changes every run.'

**Standalone of `no_grad`.** `model.train()/eval()` controls layer behavior. `torch.no_grad()` controls autograd. You need both for real validation; this drill isolates the train/eval flip.

### Exercise 1 — flip train and eval mode around dropout

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `model.train()` and `model.eval()` around a forward pass of a dropout-containing network so that dropout is stochastic during training and deterministic during evaluation.
> Keywords: train-mode, eval-mode, dropout
> ```

**KCs targeted:** `model-train-eval-flips-training-flag`, `dropout-active-only-when-training`

Implement `ex1_train_then_eval(model, x, n_repeats)`. The function isolates the train/eval flip on a network that contains `nn.Dropout`.

1. Call `model.train()`. Repeatedly forward `x` through `model` `n_repeats` times — collect every output into a list. Because dropout is active and uses fresh masks each call, consecutive outputs will DIFFER.
2. Call `model.eval()`. Repeatedly forward `x` through `model` `n_repeats` times — collect every output. Because dropout is a no-op in eval mode, every output is IDENTICAL.
3. Return `(train_outputs, eval_outputs, was_training_before, is_training_after)` where `was_training_before` is `model.training` BEFORE the function does anything, and `is_training_after` is `model.training` after `model.eval()`.

Inputs:
- `model`: an `nn.Module` containing at least one `nn.Dropout` layer with `p > 0`.
- `x`: input tensor.
- `n_repeats`: how many forward passes per mode.

The test builds a small Sequential with a `nn.Dropout(p=0.5)` in the middle and checks both the variance behavior and the `model.training` flag transitions.

In [ ]:
def ex1_train_then_eval(model, x, n_repeats):
    was_training = model.training       # snapshot first
    model.train()
    train_outs = [model(x) for _ in range(n_repeats)]
    model.eval()
    eval_outs = [model(x) for _ in range(n_repeats)]
    is_training_after = model.training
    return train_outs, eval_outs, was_training, is_training_after


<details><summary>Solution</summary>

```python
def ex1_train_then_eval(model, x, n_repeats):
    was_training = model.training       # snapshot first
    model.train()
    train_outs = [model(x) for _ in range(n_repeats)]
    model.eval()
    eval_outs = [model(x) for _ in range(n_repeats)]
    is_training_after = model.training
    return train_outs, eval_outs, was_training, is_training_after
```

**`model.training` is the toggle.** `.train(mode=True)` and `.eval()` (which is `.train(mode=False)`) walk every submodule and set `.training = mode`. Layers consult their own `self.training` flag in `forward` and branch accordingly. Dropout, BatchNorm, RMSNorm-with-tracked-stats, and any custom layer that wants train/eval differences read this flag.

**Why the flip is so common in ARENA debugging.** The most frequent ARENA training-loop bug is calling `model.eval()` during validation but forgetting to call `model.train()` again before the next epoch's training loop. The model trains on FROZEN batchnorm running stats from then on; training proceeds without error but accuracy plateaus far below what it should. Always pair them — `model.train()` at the start of the train loop, `model.eval()` at the start of validation.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()